In [1]:
import ast
import os
import re
import json
import subprocess
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import urlparse

In [2]:
def clone_repo(url: str, dest_root: str='./temp/repos/')->Path:
    """"Clone a github repo and return the local path.
    re-Clones the cleany if the destination already exist."""

    parsed=urlparse(url)
    repo_name=Path(parsed.path).stem #owner/name.git -> name
    dest=Path(dest_root)/repo_name

    if dest.exists():
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    result=subprocess.run(
        ['git', 'clone', '--depth', '1', url, str(dest)],
        capture_output=True, text=True,
    )

    if result.returncode!=0:
        raise RuntimeError(f"git clone failed: {result.stderr.strip()}")
    
    sha_result = subprocess.run(['git', '-C', str(dest), 'rev-parse', 'HEAD'],
                                 capture_output=True, text=True)
    commit_sha = sha_result.stdout.strip()[:12]  # short SHA is enough

    
    return dest, repo_name, commit_sha

In [3]:
repo_url = "https://github.com/shreeragkh/Hybrid-Search-RAG"

repo_path, repo_name, commit_sha = clone_repo(repo_url)
print(f"cloned {repo_name} ({commit_sha}) -> {repo_path}")
print(f"Files on disk: {sum(1 for _ in repo_path.rglob('*') if _.is_file())}")

cloned Hybrid-Search-RAG (48cbc7a6fdac) -> temp/repos/Hybrid-Search-RAG
Files on disk: 71


In [4]:
CODE_ONLY_MAP = {
    ".py": "python", ".js": "javascript", ".jsx": "javascript",
    ".ts": "typescript", ".tsx": "typescript", ".java": "java",
    ".go": "go", ".rb": "ruby", ".rs": "rust", ".c": "c", ".h": "c",
    ".cpp": "cpp", ".hpp": "cpp", ".cs": "csharp", ".php": "php",
}

EXCLUDE_DIRS = {".git", "node_modules", "venv", ".venv", "__pycache__",
                "dist", "build", ".next", "target", "vendor", ".idea", ".mypy_cache"}


EXCLUDE_FILENAMES = {"package-lock.json", "yarn.lock", "poetry.lock"}
EXCLUDE_PATTERNS = re.compile(r"\.min\.(js|css)$|\.d\.ts$|_pb2\.py$")
DOC_EXTENSIONS = {".md": "markdown", ".rst": "restructuredtext", ".txt": "text"}
PRIORITY_DOC_FILENAMES = {"readme.md", "readme.rst", "readme.txt", "readme"}

def discover_files(repo_dir: Path, max_file_kb: int = 500):
    files = []
    for root, dirs, filenames in os.walk(repo_dir):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS and not d.startswith(".")]
        for fn in filenames:
            if fn in EXCLUDE_FILENAMES or EXCLUDE_PATTERNS.search(fn):
                continue
            ext = Path(fn).suffix.lower()
            is_code = ext in CODE_ONLY_MAP
            is_doc = ext in DOC_EXTENSIONS or fn.lower() in PRIORITY_DOC_FILENAMES
            if not (is_code or is_doc):
                continue
            full = Path(root) / fn
            try:
                if full.stat().st_size > max_file_kb * 1024:
                    continue
            except OSError:
                continue
            files.append(full)
    return files


def chunk_markdown_file(path: Path, repo_name: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    header_pattern = re.compile(r"^#{1,3}\s+(.+)")
    starts = [i for i, line in enumerate(lines) if header_pattern.match(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, "markdown", window=80, overlap=10)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        name = header_pattern.match(lines[start]).group(1).strip()
        chunks.append(Chunk(repo_name, str(path), "markdown", "doc_section",
                             name, start + 1, end + 1, src, len(src)))
    return chunks

In [5]:

discovered = discover_files(repo_path)
print(f"Discovered {len(discovered)} chunkable files")
from collections import Counter
print(Counter(f.suffix for f in discovered).most_common())

Discovered 23 chunkable files
[('.py', 21), ('.md', 1), ('.txt', 1)]


#### Chunking

In [6]:
@dataclass
class Chunk:
    repo: str
    file_path: str
    language: str
    symbol_type: str   # "function" | "class" | "method" | "block" | "file"
    symbol_name: str
    start_line: int
    end_line: int
    content: str
    char_count: int
    chunk_id: str = ""

    def __post_init__(self):
        if not self.chunk_id:
            raw = f"{self.repo}:{self.file_path}:{self.symbol_name}:{self.start_line}-{self.end_line}"
            self.chunk_id = hashlib.sha1(raw.encode()).hexdigest()[:16]


GENERIC_FUNC_PATTERNS = {
    "javascript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "typescript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "java": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "go": re.compile(r"^\s*func\s+(\(\w+\s+\*?\w+\)\s+)?(\w+)\s*\("),
    "ruby": re.compile(r"^\s*def\s+(\w+)|^\s*class\s+(\w+)"),
    "rust": re.compile(r"^\s*(pub\s+)?fn\s+(\w+)|^\s*(pub\s+)?struct\s+(\w+)"),
    "c": re.compile(r"^\s*[\w\*]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "cpp": re.compile(r"^\s*[\w\*:<>]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "csharp": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "php": re.compile(r"^\s*function\s+(\w+)|^\s*class\s+(\w+)"),
}

MAX_CHUNK_CHARS = 3000

def split_oversized(chunk: Chunk, max_chars: int = MAX_CHUNK_CHARS):
    if chunk.char_count <= max_chars:
        return [chunk]
    lines = chunk.content.splitlines()
    out, buf, buf_start = [], [], chunk.start_line
    cur_len = 0
    for i, line in enumerate(lines):
        buf.append(line)
        cur_len += len(line) + 1
        if cur_len >= max_chars:
            src = "\n".join(buf)
            out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                              f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                              buf_start + len(buf) - 1, src, len(src)))
            buf, buf_start, cur_len = [], chunk.start_line + i + 1, 0
    if buf:
        src = "\n".join(buf)
        out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                          f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                          buf_start + len(buf) - 1, src, len(src)))
    return out

def chunk_generic_lines(path: Path, repo_name: str, text: str, language: str, window: int = 60, overlap: int = 10):
    """Sliding-window line chunks with overlap. Fallback for languages/files
    without symbol-level parsing, or when a parse attempt fails."""
    lines = text.splitlines()
    chunks = []
    i, n = 0, len(lines)
    if n == 0:
        return chunks
    while i < n:
        end = min(i + window, n)
        src = "\n".join(lines[i:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), language, "block",
                                 f"lines_{i+1}-{end}", i + 1, end, src, len(src)))
        if end == n:
            break
        i += window - overlap
    return chunks


def chunk_python_file(path: Path, repo_name: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return chunk_generic_lines(path, repo_name, text, "python")

    chunks, covered = [], set()

    def node_source(node):
        start = node.lineno
        end = getattr(node, "end_lineno", start)
        covered.update(range(start, end + 1))
        return start, end, "\n".join(lines[start - 1:end])

    for node in ast.iter_child_nodes(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "function",
                                node.name, start, end, src, len(src)))
        elif isinstance(node, ast.ClassDef):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "class",
                                node.name, start, end, src, len(src)))
            for sub in node.body:
                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    s2, e2, src2 = node_source(sub)
                    chunks.append(Chunk(repo_name, str(path), "python", "method",
                                        f"{node.name}.{sub.name}", s2, e2, src2, len(src2)))
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            start, end, src = node_source(node)
            if isinstance(node, ast.Assign) and node.targets:
                name = getattr(node.targets[0], "id", "constant")
            elif isinstance(node, ast.AnnAssign) and getattr(node.target, "id", None):
                name = node.target.id
            else:
                name = "constant"
            chunks.append(Chunk(repo_name, str(path), "python", "constant",
                                name, start, end, src, len(src)))

    leftover = [i + 1 for i in range(len(lines)) if (i + 1) not in covered]
    if leftover:
        start, end = min(leftover), max(leftover)
        src = "\n".join(lines[start - 1:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), "python", "block",
                                 "module_level", start, end, src, len(src)))
    return chunks


def chunk_generic_symbols(path: Path, repo_name: str, language: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    pattern = GENERIC_FUNC_PATTERNS.get(language)
    if pattern is None:
        return chunk_generic_lines(path, repo_name, text, language)

    starts = [i for i, line in enumerate(lines) if pattern.search(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, language)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        m = pattern.search(lines[start])
        name = next((g for g in m.groups() if g and re.match(r"^\w+$", g)), "anonymous")
        chunks.append(Chunk(repo_name, str(path), language, "function",
                             name, start + 1, end + 1, src, len(src)))
    return chunks


def chunk_file(path: Path, repo_name: str):
    ext = path.suffix.lower()
    language = CODE_ONLY_MAP.get(ext, "text")
    if language == "python":
        return chunk_python_file(path, repo_name)
    if language in GENERIC_FUNC_PATTERNS:
        return chunk_generic_symbols(path, repo_name, language)
    if ext == ".md" or path.name.lower().startswith("readme"):
        return chunk_markdown_file(path, repo_name)
    if ext in DOC_EXTENSIONS:
        text = path.read_text(encoding="utf-8", errors="ignore")
        return chunk_generic_lines(path, repo_name, text, DOC_EXTENSIONS[ext])
    text = path.read_text(encoding="utf-8", errors="ignore")
    return chunk_generic_lines(path, repo_name, text, language)


def chunk_repo(repo_dir: Path, repo_name: str):
    all_chunks = []
    for f in discover_files(repo_dir):
        for c in chunk_file(f,repo_name):
            all_chunks.extend(split_oversized(c))
    return all_chunks


In [7]:
import hashlib

chunks = chunk_repo(repo_path, repo_name)

print(f"Total chunks: {len(chunks)}")
from collections import Counter
print("By symbol_type:", Counter(c.symbol_type for c in chunks))
print("By language:   ", Counter(c.language for c in chunks))
print(f"Avg chunk size: {sum(c.char_count for c in chunks) / len(chunks):.0f} chars")

print("\nSample chunks:")
for c in chunks[:20]:
    print(f"  [{c.language:10}] {c.symbol_type:8} {c.symbol_name:25} "
          f"{Path(c.file_path).name}:{c.start_line}-{c.end_line}")


Total chunks: 171
By symbol_type: Counter({'method': 37, 'block': 35, 'function': 30, 'constant': 28, 'class': 21, 'doc_section': 20})
By language:    Counter({'python': 150, 'markdown': 20, 'text': 1})
Avg chunk size: 843 chars

Sample chunks:
  [markdown  ] doc_section Hybrid Search RAG Application README.md:1-5
  [markdown  ] doc_section 🏗️ Architecture Overview  README.md:7-53
  [markdown  ] doc_section ✨ Key Features            README.md:55-71
  [markdown  ] doc_section 📁 Project Structure       README.md:73-120
  [markdown  ] doc_section 🛠️ Tech Stack & Models Used README.md:122-134
  [markdown  ] doc_section 🚀 Quick Start             README.md:136-136
  [markdown  ] doc_section 1. Prerequisites          README.md:138-143
  [markdown  ] doc_section 2. Environment Setup      README.md:145-151
  [markdown  ] doc_section Create virtual environment README.md:153-155
  [markdown  ] doc_section Install dependencies      README.md:157-159
  [markdown  ] doc_section 3. Environment Variab

In [8]:
OUT_PATH = f"./temp/repos/{repo_name}/chunks-{repo_name}.jsonl"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c)) + "\n")

print(f"Wrote {len(chunks)} chunks to {OUT_PATH}")


Wrote 171 chunks to ./temp/repos/Hybrid-Search-RAG/chunks-Hybrid-Search-RAG.jsonl


#### Embedding and Db

In [9]:
import os
from dataclasses import asdict
from dotenv import load_dotenv
from astrapy import DataAPIClient
from astrapy.constants import VectorMetric
from sentence_transformers import SentenceTransformer
from astrapy.info import CollectionDefinition

load_dotenv()

# Load the embedding model locally (runs on your machine, free, no API calls)
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Initialize the client
client = DataAPIClient()
db = client.get_database(
    api_endpoint=os.getenv("API_ENDPOINT"),
    token=os.getenv("API_TOKEN"),
)

definition = (
    CollectionDefinition.builder()
    .with_vector_dimension(1024)
    .with_vector_metric(VectorMetric.COSINE)
    .build()
)

# Drop the old collection if it exists — it was created with a `service` block,
# which is incompatible with bringing your own vectors. Must recreate clean.
if "repo_context" in db.list_collection_names():
    db.drop_collection("repo_context")

# Create collection WITHOUT a service block — no Astra-side embedding provider needed
collection = db.create_collection(
    "repo_context",
    definition=definition
)

# Compute embeddings locally
texts = [c.content for c in chunks]
vectors = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
).tolist()

# Prepare documents with $vector (pre-computed), not $vectorize
documents = [{"_id": c.chunk_id, "$vector": vec, **asdict(c)} for vec, c in zip(vectors, chunks)]

# Insert chunks — no embedding provider call per-batch anymore, so no timeouts,
# can use a larger batch size and don't need retry logic for provider timeouts
batch_size = 50
all_inserted = []
for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    result = collection.insert_many(batch, request_timeout_ms=30000)
    all_inserted.extend(result.inserted_ids)
    print(f"Batch {i // batch_size + 1}: inserted {len(result.inserted_ids)}")

print(f"\nSuccessfully inserted {len(all_inserted)} chunks into Astra DB!")
print(f"Collections in Astra DB: {db.list_collection_names()}")

/home/shreerag/Desktop/shreeragkh/Repo-context-copilot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 6/6 [02:58<00:00, 29.73s/it]


Batch 1: inserted 50
Batch 2: inserted 50
Batch 3: inserted 50
Batch 4: inserted 21

Successfully inserted 171 chunks into Astra DB!
Collections in Astra DB: ['repo_context']


In [10]:
import json
import logging
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import bm25s

logger = logging.getLogger(__name__)


class BM25Retriever:
    """
    Wraps bm25s.BM25 to support:
      - building an index from a list of chunk dicts or dataclasses (text + metadata)
      - persisting the index and metadata to disk, scoped per repo via index_dir
      - reloading without re-tokenizing the corpus
      - querying with scores, metadata, and chunk_id attached (for RRF fusion
        against vector search results keyed on the same chunk_id)
    """

    def __init__(self, index_dir: str | Path = "bm25_index"):
        self.index_dir = Path(index_dir)
        self.retriever: bm25s.BM25 | None = None
        self.corpus: list[str] = []
        self.metadata: list[dict[str, Any]] = []

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _to_dicts(chunks: list[Any]) -> list[dict[str, Any]]:
        """Accept a list of dicts or dataclass instances (e.g. Chunk) transparently."""
        return [asdict(c) if is_dataclass(c) else c for c in chunks]

    # ------------------------------------------------------------------
    # Build
    # ------------------------------------------------------------------
    def build(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Build the BM25 index from scratch.

        Args:
            chunks: list of dicts or dataclass instances, each containing at
                    least `text_key` (e.g. the Chunk dataclass's `content` field,
                    and ideally a `chunk_id` field for fusion with vector results).
                    All other keys are stored as metadata and returned
                    alongside results at query time.
            text_key: the dict key holding the chunk's raw text. Defaults to
                    "content" to match the project's Chunk dataclass.
        """
        chunk_dicts = self._to_dicts(chunks)
        if not chunk_dicts:
            raise ValueError("Cannot build BM25 index from an empty chunk list.")

        self.corpus = [c[text_key] for c in chunk_dicts]
        self.metadata = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        logger.info("Tokenizing %d chunks for BM25 indexing...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)

        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)
        logger.info("BM25 index built with %d documents.", len(self.corpus))

    # ------------------------------------------------------------------
    # Incremental-ish rebuild (bm25s has no true incremental add;
    # this re-tokenizes the full corpus with new chunks appended)
    # ------------------------------------------------------------------
    def add(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Append new chunks and rebuild the index. bm25s does not support
        true incremental indexing, so this re-indexes the full corpus.
        Fine for periodic batch updates (e.g. re-ingesting a repo); avoid
        calling this per-request.
        """
        chunk_dicts = self._to_dicts(chunks)
        new_texts = [c[text_key] for c in chunk_dicts]
        new_meta = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        self.corpus.extend(new_texts)
        self.metadata.extend(new_meta)

        logger.info("Rebuilding BM25 index with %d total documents...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)
        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)

    # ------------------------------------------------------------------
    # Persistence
    # ------------------------------------------------------------------
    def save(self) -> None:
        """Persist the BM25 index, corpus, and metadata to self.index_dir."""
        if self.retriever is None:
            raise RuntimeError("No index to save. Call build() first.")

        self.index_dir.mkdir(parents=True, exist_ok=True)

        # bm25s handles the index + corpus itself
        self.retriever.save(str(self.index_dir), corpus=self.corpus)

        # metadata isn't tracked by bm25s, so store it ourselves
        meta_path = self.index_dir / "metadata.json"
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metadata, f)

        logger.info("Saved BM25 index and metadata to %s", self.index_dir)

    def load(self) -> None:
        """Load a previously saved index, corpus, and metadata from disk."""
        if not self.index_dir.exists():
            raise FileNotFoundError(f"No index found at {self.index_dir}")

        self.retriever = bm25s.BM25.load(str(self.index_dir), load_corpus=True)

        meta_path = self.index_dir / "metadata.json"
        if meta_path.exists():
            with open(meta_path, "r", encoding="utf-8") as f:
                self.metadata = json.load(f)
        else:
            logger.warning("No metadata.json found at %s; metadata will be empty.", self.index_dir)
            self.metadata = [{} for _ in range(len(self.retriever.corpus))]

        self.corpus = [doc["text"] if isinstance(doc, dict) else doc for doc in self.retriever.corpus]

        # Detach the corpus from the bm25s object so retrieve() always
        # returns plain indices rather than document text/dicts. This keeps
        # query() lookups simple and correct even with duplicate chunk text.
        self.retriever.corpus = None

        logger.info("Loaded BM25 index with %d documents from %s", len(self.corpus), self.index_dir)

    # ------------------------------------------------------------------
    # Query
    # ------------------------------------------------------------------
    def query(self, query_text: str, k: int = 10) -> list[dict[str, Any]]:
        """
        Retrieve top-k chunks for a query.

        Returns:
            list of dicts: {"text": ..., "score": ..., "chunk_id": ..., "metadata": {...}}
            sorted by descending BM25 score. `chunk_id` is lifted out of metadata
            (if present) to the top level so it can be joined directly against
            vector search results in an RRF fusion step.
        """
        if self.retriever is None:
            raise RuntimeError("Index not built or loaded. Call build() or load() first.")

        k = min(k, len(self.corpus))
        if k == 0:
            return []

        query_tokens = bm25s.tokenize(query_text, show_progress=False)
        doc_indices, scores = self.retriever.retrieve(query_tokens, k=k, show_progress=False)

        results = []
        for idx, score in zip(doc_indices[0], scores[0]):
            idx = int(idx)
            meta = self.metadata[idx] if idx < len(self.metadata) else {}
            results.append({
                "text": self.corpus[idx],
                "score": float(score),
                "chunk_id": meta.get("chunk_id"),
                "metadata": meta,
            })
        return results

    def __len__(self) -> int:
        return len(self.corpus)

In [11]:
bm25=BM25Retriever(index_dir=f"./bm25_index/{repo_name}")
bm25.build(chunks)
bm25.save()

#### Retreiever

In [12]:
from typing import List, Dict, Any


class VectorRetriever:
    """Handles query-based semantic retrieval from the Astra DB vector store."""

    def __init__(self, collection, model):
        """
        Initialize the retriever pipeline for Astra DB.

        Args:
            collection: an astrapy Collection with pre-computed $vector fields
                        (see ingestion pipeline — chunks were embedded locally
                        with sentence-transformers and inserted as $vector).
            model: the same SentenceTransformer instance used at ingestion time.
                   Must match exactly, or query/document vectors won't be comparable.
        """
        self.collection = collection
        self.model = model

    def query(self, query_text: str, k: int = 5) -> List[Dict[str, Any]]:
        """Helper to match the query interface of other retrievers (e.g. BM25Retriever)."""
        return self.retrieve(query_text, top_k=k)

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant chunks for a query via vector similarity search.

        Args:
            query: query from the user
            top_k: number of top results to return
            score_threshold: minimum similarity score threshold (0-1, cosine)

        Returns:
            List of dicts: {"chunk_id", "text", "content", "metadata",
            "score", "rank"} — shaped to match BM25Retriever.query() output
            so both can be merged directly in RRF fusion.
        """
        try:
            query_vector = self.model.encode(
                [query], normalize_embeddings=True
            ).tolist()[0]

            results = self.collection.find(
                sort={"$vector": query_vector},
                limit=top_k,
                include_similarity=True,
            )

            retrieved_docs = []
            for i, doc in enumerate(results):
                similarity_score = doc.get("$similarity", 0.0)
                if similarity_score < score_threshold:
                    continue

                content = doc.get("content", "")
                metadata = {
                    k: v for k, v in doc.items()
                    if k not in ("_id", "$vector", "$similarity", "content")
                }

                retrieved_docs.append({
                    "id": doc.get("_id"),
                    "chunk_id": doc.get("chunk_id", doc.get("_id")),
                    "text": content,
                    "content": content,
                    "metadata": metadata,
                    "score": similarity_score,
                    "similarity_score": similarity_score,
                    "rank": i + 1,
                })

            print(f"Retrieved documents: {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Usage
vector_retrieval = VectorRetriever(collection, model)  # `model` = your SentenceTransformer instance

#### HybridSearch

In [13]:
from __future__ import annotations
import logging
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
from typing import Any, Callable

logger = logging.getLogger(__name__)


class HybridSearchError(Exception):
    """Raised only when BOTH retrievers fail — total retrieval failure."""


def _reciprocal_rank_fusion(
    bm25_results: list[dict],
    vector_results: list[dict],
    bm25_weight: float,
    vector_weight: float,
    rrf_k: int = 60,
    id_key: str = "chunk_id",
) -> list[dict]:

    fused_docs = {}

    def get_id(doc):
        # chunk_id is exposed at the TOP LEVEL by both BM25Retriever and
        # VectorRetriever (see their query() implementations), so check
        # there first. Fall back to metadata, then raw text as a last resort
        # for any retriever that doesn't provide a stable chunk_id.
        if doc.get(id_key):
            return str(doc[id_key])
        meta = doc.get("metadata") or {}
        if id_key in meta and meta[id_key]:
            return str(meta[id_key])
        text = doc.get("text") or doc.get("content") or ""
        return text.strip()

    for rank, doc in enumerate(bm25_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("score", 0.0)

        fused_docs[doc_id] = {
            "chunk_id": doc.get(id_key) or doc_id,
            "text": text,
            "metadata": metadata,
            "bm25_score": score,
            "vector_score": 0.0,
            "bm25_rrf": bm25_weight * (1.0 / (rrf_k + (rank + 1))),
            "vector_rrf": 0.0,
        }

    for rank, doc in enumerate(vector_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("similarity_score") or doc.get("score") or 0.0

        if doc_id in fused_docs:
            fused_docs[doc_id]["vector_score"] = score
            fused_docs[doc_id]["vector_rrf"] = vector_weight * (1.0 / (rrf_k + (rank + 1)))
            if not fused_docs[doc_id]["metadata"] and metadata:
                fused_docs[doc_id]["metadata"] = metadata
            if not fused_docs[doc_id]["text"] and text:
                fused_docs[doc_id]["text"] = text
        else:
            fused_docs[doc_id] = {
                "chunk_id": doc.get(id_key) or doc_id,
                "text": text,
                "metadata": metadata,
                "bm25_score": 0.0,
                "vector_score": score,
                "bm25_rrf": 0.0,
                "vector_rrf": vector_weight * (1.0 / (rrf_k + (rank + 1))),
            }

    output = []
    for doc_id, info in fused_docs.items():
        fused_score = info["bm25_rrf"] + info["vector_rrf"]
        output.append({
            "chunk_id": info["chunk_id"],
            "text": info["text"],
            "metadata": info["metadata"],
            "fused_score": fused_score,
            "bm25_score": info["bm25_score"],
            "vector_score": info["vector_score"],
        })

    output.sort(key=lambda x: x["fused_score"], reverse=True)
    return output


class HybridSearch:
    """Handles query-based hybrid search: BM25 + vector retrieval fused via RRF."""

    def __init__(self, bm25_retriever, vector_retriever):
        self.bm25_retriever = bm25_retriever
        self.vector_retriever = vector_retriever

    def hybrid_retrieval(
        self,
        query_text: str,
        k: int = 10,
        fetch_k: int = 25,
        bm25_weight: float = 0.4,
        vector_weight: float = 0.6,
        rrf_k: int = 60,
        id_key: str = "chunk_id",
        metadata_filter: Callable[[dict], bool] | None = None,
        timeout_s: float = 15.0,
    ) -> list[dict[str, Any]]:
        """
        Runs BM25 and vector retrieval in parallel, fuses with RRF, and returns top-k.

        Args:
            query_text: user's query, e.g. "how does the auth middleware work?"
            k: number of results to return after fusion.
            fetch_k: candidates pulled from EACH retriever before fusion.
            bm25_weight / vector_weight: RRF weighting between the two signals.
            rrf_k: RRF damping constant (60 is the standard default).
            id_key: field used as the stable dedup key across both retrievers.
                    Defaults to "chunk_id", set on every Chunk at ingestion time
                    and returned at the top level by both retrievers.
            metadata_filter: optional predicate applied after fusion, e.g.
                    lambda m: m.get("language") == "python".
            timeout_s: max seconds to wait for EACH retriever before treating
                    it as failed and falling back to the other.

        Returns:
            List of {"chunk_id", "text", "metadata", "fused_score",
            "bm25_score", "vector_score"} sorted by fused_score descending.

        Raises:
            HybridSearchError if both retrievers fail.
        """
        start = time.monotonic()
        bm25_results, vector_results = self._run_retrievers_with_fallback(
            query_text, fetch_k, timeout_s
        )

        fused = _reciprocal_rank_fusion(
            bm25_results, vector_results, bm25_weight, vector_weight, rrf_k, id_key
        )

        if metadata_filter is not None:
            fused = [r for r in fused if metadata_filter(r.get("metadata", {}))]
        results = fused[:k]

        logger.info(
            "hybrid_search query=%r bm25_hits=%d vector_hits=%d fused=%d returned=%d latency_ms=%.0f",
            query_text, len(bm25_results), len(vector_results), len(fused), len(results),
            (time.monotonic() - start) * 1000,
        )
        return results

    def _run_retrievers_with_fallback(
        self, query_text: str, fetch_k: int, timeout_s: float
    ) -> tuple[list[dict], list[dict]]:
        """Run both retrievers concurrently; a failure/timeout in one degrades
        gracefully to results from the other instead of raising."""

        def safe_call(fn, name: str) -> list[dict]:
            try:
                return fn(query_text, k=fetch_k)
            except Exception:
                logger.exception("Retriever %s failed", name)
                return []

        with ThreadPoolExecutor(max_workers=2) as executor:
            bm25_future = executor.submit(safe_call, self.bm25_retriever.query, "bm25")
            vector_future = executor.submit(safe_call, self.vector_retriever.query, "vector")

            try:
                bm25_results = bm25_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("BM25 retriever timed out after %.1fs", timeout_s)
                bm25_results = []

            try:
                vector_results = vector_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("Vector retriever timed out after %.1fs", timeout_s)
                vector_results = []

        if not bm25_results and not vector_results:
            raise HybridSearchError(f"Both retrievers failed or timed out for query: {query_text!r}")
        return bm25_results, vector_results


# Usage — bm25_retriever from BM25Retriever, vector_retriever from VectorRetriever
hybrid_search = HybridSearch(bm25, vector_retrieval)
hybrid_search.hybrid_retrieval("how does the authentication middleware work?")

Retrieved documents: 25 documents (after filtering)


[{'chunk_id': 'b36827bfe36fe682',
  'text': "## 🔐 Firebase Auth Setup\n\nTo set up Google Sign-In for Admin authentication:\n\n1. Create a Firebase Project in [Firebase Console](https://console.firebase.google.com/).\n2. Enable **Google Sign-In** under **Authentication → Sign-in method**.\n3. Add `localhost` under **Authentication → Settings → Authorized domains**.\n4. Update `.env` with your project's `FIREBASE_API_KEY`, `FIREBASE_AUTH_DOMAIN`, and `FIREBASE_PROJECT_ID`.\n5. Set `ADMIN_EMAIL` in `.env` to your authorized Google email address.\n\n---",
  'metadata': {'repo': 'Hybrid-Search-RAG',
   'file_path': 'temp/repos/Hybrid-Search-RAG/README.md',
   'language': 'markdown',
   'symbol_type': 'doc_section',
   'symbol_name': '🔐 Firebase Auth Setup',
   'start_line': 225,
   'end_line': 235,
   'char_count': 501,
   'chunk_id': 'b36827bfe36fe682'},
  'fused_score': 0.015380906460945035,
  'bm25_score': 2.3985211849212646,
  'vector_score': 0.7955601},
 {'chunk_id': 'ca2e416d26e7da24

#### Reranker

In [14]:
from __future__ import annotations
 
import logging
import time
from typing import Any, Protocol
 
logger = logging.getLogger(__name__)
 
DEFAULT_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Stronger, slower alternative: "BAAI/bge-reranker-base" or "BAAI/bge-reranker-large"
 
 
class ScoringBackend(Protocol):
    """Minimal interface a reranking backend must satisfy."""
    def predict(self, pairs: list[tuple[str, str]]) -> list[float]: ...
 
 
class RerankerError(Exception):
    """Raised when reranking fails and no safe fallback is possible."""
 
 
class Reranker:
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        batch_size: int = 32,
        device: str | None = None,
        backend: ScoringBackend | None = None,
    ):
        """
        Args:
            model_name: HuggingFace cross-encoder model id. Ignored if
                        `backend` is supplied.
            batch_size: pairs per forward pass. Tune to your GPU/CPU memory.
            device: "cuda", "cpu", or None to let sentence-transformers pick.
            backend: inject a custom scoring backend (e.g. a Cohere Rerank
                     wrapper) instead of loading a local model. Must expose
                     .predict(list[(query, doc_text)]) -> list[float].
        """
        self.batch_size = batch_size
        self.model_name = model_name
 
        if backend is not None:
            self.backend = backend
        else:
            self.backend = self._load_local_model(model_name, device)
 
    @staticmethod
    def _load_local_model(model_name: str, device: str | None):
        try:
            from sentence_transformers import CrossEncoder
        except ImportError as e:
            raise ImportError(
                "sentence-transformers is required for local reranking. "
                "Install with: pip install sentence-transformers --break-system-packages"
            ) from e
 
        logger.info("Loading cross-encoder reranker model: %s", model_name)
        model = CrossEncoder(model_name, device=device)
        return model
 
    def rerank(
        self,
        query: str,
        candidates: list[dict[str, Any]],
        top_n: int = 5,
        min_score: float | None = None,
        text_key: str = "text",
        fallback_on_error: bool = True,
    ) -> list[dict[str, Any]]:
        """
        Score each candidate against the query and return the top_n,
        re-sorted by cross-encoder relevance score.
 
        Args:
            query: the user query.
            candidates: list of dicts (as returned by hybrid_search), each
                        containing at least `text_key`.
            top_n: number of results to return after reranking.
            min_score: optional threshold; candidates scoring below this
                       are dropped even if within top_n. Use this to avoid
                       feeding clearly-irrelevant context to the LLM when
                       retrieval had a bad day.
            text_key: dict key holding each candidate's text.
            fallback_on_error: if True and scoring fails, return the
                       original candidates truncated to top_n rather than
                       raising — keeps the pipeline degrading gracefully
                       instead of hard-failing generation.
 
        Returns:
            List of candidate dicts (original fields preserved) with an
            added "rerank_score" key, sorted descending, length <= top_n.
        """
        if not candidates:
            return []
 
        start = time.monotonic()
        pairs = [(query, c[text_key]) for c in candidates]
 
        try:
            scores = self._score_in_batches(pairs)
        except Exception:
            logger.exception("Reranking failed for query=%r (%d candidates)", query, len(candidates))
            if fallback_on_error:
                logger.warning("Falling back to pre-rerank order (no cross-encoder scores applied).")
                return [{**c, "rerank_score": c.get("fused_score", 0.0)} for c in candidates[:top_n]]
            raise RerankerError(f"Reranking failed for query: {query!r}")
 
        scored = [
            {**cand, "rerank_score": float(score)}
            for cand, score in zip(candidates, scores)
        ]
        scored.sort(key=lambda c: c["rerank_score"], reverse=True)
 
        if min_score is not None:
            scored = [c for c in scored if c["rerank_score"] >= min_score]
 
        results = scored[:top_n]
 
        logger.info(
            "rerank query=%r candidates=%d returned=%d top_score=%.4f latency_ms=%.0f",
            query, len(candidates), len(results),
            results[0]["rerank_score"] if results else float("nan"),
            (time.monotonic() - start) * 1000,
        )
        return results
 
    def _score_in_batches(self, pairs: list[tuple[str, str]]) -> list[float]:
        scores: list[float] = []
        for i in range(0, len(pairs), self.batch_size):
            batch = pairs[i : i + self.batch_size]
            batch_scores = self.backend.predict(batch)
            scores.extend(float(s) for s in batch_scores)
        return scores


In [15]:
re_ranker=Reranker()
re_ranker

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2510.89it/s]


#### RAG PipeLine

In [164]:
import re
import markdown
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    llm = ChatGroq(model="qwen/qwen3.6-27b", reasoning_format="hidden", temperature=0.1, max_tokens=512)
else:
    print("GROQ_API_KEY not found in environment. Falling back to FakeMessagesListChatModel for mock responses.")
    llm = FakeMessagesListChatModel(responses=[AIMessage(content="Mock response — no GROQ_API_KEY set.")])


def _empty_result(message: str, complexity="UNKNOWN", complexity_conf=0.0,
                   complexity_reason="", fetch_k=0, return_context=False):
    """Keeps the return schema identical across every exit path."""
    out = {
        'answer': message, 'sources': [], 'confidence': 0.0,
        'complexity': complexity, 'complexity_confidence': complexity_conf,
        'complexity_reason': complexity_reason, 'retrieval_k': fetch_k,
        'final_chunk_count': 0,
    }
    if return_context:
        out['context'] = ""
    return out


def ragPipeline(query, hybrid_search=None, reranker=None, llm=None, top_k=None,
                 top_n=None, min_score=0.2, return_context=False, use_adaptive=True):
    """RAG Pipeline with adaptive retrieval — returns answer, sources,
    confidence, and complexity metadata; optionally full context."""

    if reranker is not None and (hasattr(reranker, 'invoke') and not hasattr(reranker, 'rerank')):
        llm = reranker
        reranker = None

    if hybrid_search is None:
        hybrid_search = globals().get('hybrid_search')
        if hybrid_search is None:
            raise ValueError("No hybrid_search instance provided or found in global scope.")

    if llm is None:
        llm = globals().get('llm')
        if llm is None:
            raise ValueError("No LLM instance provided or found in global scope.")

    if reranker is None:
        reranker = globals().get('re_ranker') or globals().get('reranker')

    complexity, complexity_conf, complexity_reason = classify_complexity(query, llm)
    fetch_k = top_k or RETRIEVAL_BUDGET.get(complexity, RETRIEVAL_BUDGET["MEDIUM"])

    results = hybrid_search.hybrid_retrieval(query, k=fetch_k)

    bm25_weight, vector_weight, rrf_k = 0.4, 0.6, 60
    max_rrf_score = (bm25_weight + vector_weight) / (rrf_k + 1)

    results = [doc for doc in results if (doc.get('fused_score', 0.0) / max_rrf_score) >= min_score]

    if not results:
        return _empty_result('No relevant answer found.', complexity, complexity_conf,
                              complexity_reason, fetch_k, return_context)

    if reranker is not None:
        rerank_top_n = top_n or fetch_k
        results = reranker.rerank(query, results, top_n=rerank_top_n)
        if use_adaptive:
            results = adaptive_cutoff(results, complexity=complexity)

    # Always trim to fit the model's context window, regardless of use_adaptive.
    prompt_overhead = count_tokens(query) + 80
    results = trim_to_token_budget(results,
                                    tpm_limit=MODEL_CONTEXT_WINDOW,
                                    reserved_output_tokens=512,
                                    prompt_overhead_tokens=prompt_overhead,
                                    safety_margin=200)

    if not results:
        return _empty_result('No relevant answer found after rerank.', complexity, complexity_conf,
                              complexity_reason, fetch_k, return_context)

    context = "\n\n".join([doc['text'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc.get('fused_score', 0.0) / max_rrf_score,
        'preview': doc['text'][:120] + '...'
    } for doc in results]

    confidence = max(doc.get('fused_score', 0.0) / max_rrf_score for doc in results)

    prompt = f"""You are a helpful assistant answering questions about a codebase.
                Answer directly and concisely in 1-3 sentences. Do not add greetings, disclaimers, or unnecessary preamble.

                Context:
                {context}

                Question: {query}

                Answer:"""

    try:
        response = llm.invoke([prompt])
        content = response.content
    except Exception as e:
        return _empty_result(f'Generation failed: {e}', complexity, complexity_conf,
                              complexity_reason, fetch_k, return_context)

    content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
    html = markdown.markdown(content)
    plain_text = re.sub(r'<[^>]*>', '', html)

    output = {
        'answer': plain_text.strip(),
        'sources': sources,
        'confidence': confidence,
        'complexity': complexity,
        'complexity_confidence': complexity_conf,
        'complexity_reason': complexity_reason,
        'retrieval_k': fetch_k,
        'final_chunk_count': len(results),
    }

    if return_context:
        output['context'] = context
    return output

In [165]:
COMPLEXITY_PROMPT = """
You are a query complexity classifier for a code repository.

Your task is to determine how much repository context is likely
required to answer the user's query.

Classify the query into exactly one of these levels:

LOW:
- Can probably be answered from one file, function, class, or
  small local section.
- Does not require significant cross-file reasoning.

MEDIUM:
- Requires understanding multiple related files, functions,
  or components.
- May require following a limited data or execution flow.

HIGH:
- Requires understanding multiple components or subsystems.
- Requires tracing a multi-step execution or data flow.
- Requires architectural or dependency reasoning.
- Requires understanding how several parts of the repository interact.

Consider:
1. Number of components involved
2. Number of files likely to be required
3. Whether the query requires tracing a flow
4. Whether cross-file reasoning is required
5. Whether architectural reasoning is required
6. Whether multiple steps need to be understood
7. Whether the query asks for comparison or impact analysis

Do not classify based only on query length.

Return ONLY valid JSON:

{{
    "complexity": "LOW | MEDIUM | HIGH",
    "confidence": 0.0,
    "reason": "Brief explanation"
}}

User query:
{query}
"""

#### Classify complexity

In [166]:
import re

def classify_complexity_heuristic(query: str) -> str | None:
    """Fast, free heuristic. Returns None if uncertain -> triggers LLM fallback."""
    q = query.lower().strip()
    word_count = len(q.split())

    high_signals = [" and ", " then ", "trace", "flow", "end to end", "end-to-end",
                     "architecture", "impact", "compare", "across", "interact"]
    low_signals = ["what is", "where is", "define", "which file", "what does"]

    high_hits = sum(1 for s in high_signals if s in q)
    low_hits = sum(1 for s in low_signals if s in q)

    if word_count <= 6 and low_hits > 0 and high_hits == 0:
        return "LOW"
    if high_hits >= 2 or word_count > 25:
        return "HIGH"
    if high_hits == 0 and low_hits == 0 and word_count <= 15:
        return None  # ambiguous -> let the LLM decide
    return "MEDIUM"


def classify_complexity(query: str, llm) -> tuple[str, float, str]:
    """Heuristic first; LLM fallback only when the heuristic is unsure."""
    heuristic_result = classify_complexity_heuristic(query)
    if heuristic_result is not None:
        return heuristic_result, 1.0, "heuristic"

    prompt = COMPLEXITY_PROMPT.format(query=query)
    response = llm.invoke(prompt)
    try:
        data = json.loads(response.content)
        return data["complexity"], data.get("confidence", 0.5), data.get("reason", "llm_fallback")
    except (json.JSONDecodeError, KeyError):
        return "MEDIUM", 0.0, "classification_failed"

#### adaptive cut off

In [167]:
DROPOFF_BY_COMPLEXITY = {"LOW": 0.45, "MEDIUM": 0.55, "HIGH": 0.70}

def adaptive_cutoff(reranked_results: list[dict], min_keep: int = 1,
                     max_keep: int = 15, complexity="MEDIUM") -> list[dict]:
    """
    Walk down reranked_score-sorted results and stop at the first sharp
    relative drop between consecutive scores. Assumes results are already
    sorted descending by 'rerank_score' (true for Reranker.rerank() output).
    """

    dropoff_ratio = DROPOFF_BY_COMPLEXITY.get(complexity, 0.55)
    min_keep = {"LOW": 2, "MEDIUM": 3, "HIGH": 5}.get(complexity, min_keep)
    if not reranked_results:
        return []
    if len(reranked_results) <= min_keep:
        return reranked_results

    kept = [reranked_results[0]]
    for i in range(1, min(len(reranked_results), max_keep)):
        if len(kept)>=min_keep:
            prev_score = reranked_results[i - 1]["rerank_score"]
            curr_score = reranked_results[i]["rerank_score"]
            # Cross-encoder scores can be negative/near-zero raw logits —
            # use absolute gap normalized by magnitude instead of a plain
            # ratio, which breaks when prev_score <= 0.
            denom = max(abs(prev_score), 1e-6)
            drop = (prev_score - curr_score) / denom
            if drop > dropoff_ratio and len(kept) >= min_keep:
                break
        kept.append(reranked_results[i])
    return kept

In [168]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

# Set budget conservatively (3500 tokens) to ensure prompt + response stay well under
# Groq's 8,000 TPM limit, even when running baseline & adaptive back-to-back.
MODEL_CONTEXT_WINDOW = 3500

def count_tokens(text: str) -> int:
    # Qwen tokenizer produces ~20% more tokens than cl100k_base for code.
    # Multiply by 1.25 to stay strictly conservative.
    return int(len(enc.encode(text)) * 1.25)

def trim_to_token_budget(results, context_token_budget=None,
                          tpm_limit=MODEL_CONTEXT_WINDOW,
                          reserved_output_tokens=512,
                          prompt_overhead_tokens=150,
                          safety_margin=300):
    """Trim results so the full request fits within the Groq TPM limit.

    Budget = tpm_limit - reserved_output_tokens - prompt_overhead_tokens - safety_margin"""
    if context_token_budget is None:
        context_token_budget = tpm_limit - reserved_output_tokens - prompt_overhead_tokens - safety_margin
    context_token_budget = max(context_token_budget, 200)  # floor
    kept, total = [], 0
    for r in results:
        t = count_tokens(r['text'])
        if kept and total + t > context_token_budget:
            break
        kept.append(r)
        total += t
    return kept

In [169]:
RETRIEVAL_BUDGET = {
    "LOW": 10,
    "MEDIUM": 20,
    "HIGH": 30
}

In [137]:
user_query = "What is default ADMIN_EMAIL"
complexity, complexity_confidence, reason = classify_complexity(user_query, llm)

initial_k = RETRIEVAL_BUDGET.get(
    complexity,
    RETRIEVAL_BUDGET["MEDIUM"]
)

In [68]:
print("\n===== QUERY COMPLEXITY =====")
print("Complexity:", complexity)
print("Confidence:", complexity_confidence)
print("Reason:", reason)
print("Initial Retrieval K:", initial_k)


===== QUERY COMPLEXITY =====
Complexity: LOW
Confidence: 1.0
Reason: heuristic
Initial Retrieval K: 10


In [69]:
result = ragPipeline(
    user_query,
    hybrid_search=hybrid_search,
    reranker=re_ranker,
    llm=llm,
    top_k=initial_k
)

Retrieved documents: 25 documents (after filtering)
finish_reason: stop
raw content: 'The default `ADMIN_EMAIL` is `"shreerag99@gmail.com"`.'
additional_kwargs: {}


In [70]:
print(result)
print("Answer:", result['answer'])
print("Complexity:", result['complexity'], f"(confidence={result['complexity_confidence']})")
print("Reason:", result['complexity_reason'])
print("Retrieval K:", result['retrieval_k'], "-> Final chunks used:", result['final_chunk_count'])

{'answer': 'The default ADMIN_EMAIL is "shreerag99@gmail.com".', 'sources': [{'source': 'unknown', 'page': 'unknown', 'score': 1.0, 'preview': 'ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "shreerag99@gmail.com")...'}, {'source': 'unknown', 'page': 'unknown', 'score': 0.9715725806451612, 'preview': 'def is_admin(email: str) -> bool:\n    """Check if the email matches the hardcoded admin email."""\n    return email.strip...'}], 'confidence': 1.0, 'complexity': 'LOW', 'complexity_confidence': 1.0, 'complexity_reason': 'heuristic', 'retrieval_k': 10, 'final_chunk_count': 2}
Answer: The default ADMIN_EMAIL is "shreerag99@gmail.com".
Complexity: LOW (confidence=1.0)
Reason: heuristic
Retrieval K: 10 -> Final chunks used: 2


#### Cache layer

In [170]:
import hashlib
import json
import redis
import os

redis_client = redis.Redis(
    host=os.environ.get("REDIS_HOST", "localhost"),
    port=int(os.environ.get("REDIS_PORT", 6379)),
    password=os.environ.get("REDIS_PASSWORD"),
    decode_responses=True,
)

CACHE_TTL_SECONDS = 60 * 60 * 24  # 24h; tune based on how often the repo changes

def _cache_key(repo_name: str, query: str) -> str:
    raw = f"{repo_name}:{commit_sha}:{query.strip().lower()}"
    return "ragcache:" + hashlib.sha256(raw.encode()).hexdigest()

def ragPipeline_cached(query, repo_name, commit_sha, hybrid_search=None, reranker=None, llm=None,
                        top_k=None, top_n=None, min_score=0.2, return_context=False,
                        use_cache=True):
    key = _cache_key(repo_name, commit_sha, query)

    if use_cache:
        try:
            cached = redis_client.get(key)
            if cached:
                result = json.loads(cached)
                result["_cache_hit"] = True
                return result
        except redis.RedisError as e:
            print(f"[cache] Redis unavailable, skipping cache read: {e}")

    result = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker, llm=llm,
                          top_k=top_k, top_n=top_n, min_score=min_score,
                          return_context=return_context)
    result["_cache_hit"] = False

    if use_cache:
        try:
            redis_client.setex(key, CACHE_TTL_SECONDS, json.dumps(result))
        except redis.RedisError as e:
            print(f"[cache] Redis unavailable, skipping cache write: {e}")

    return result

#### Evaluation Harness

In [171]:
import csv
import time
import re

EVAL_SET = [
    # ---- LOW: single file / localized query ----
    {"query": "What language is this repository written in?", "expected_complexity": "LOW"},
    {"query": "Where is the FastAPI entry point defined?", "expected_complexity": "LOW"},
    {"query": "What model is used for generating document embeddings?", "expected_complexity": "LOW"},
    {"query": "What cross-encoder model is used for reranking?", "expected_complexity": "LOW"},
    {"query": "What is the default TTL for cached retrieval results in Redis?", "expected_complexity": "LOW"},
    {"query": "What weights are used for BM25 and vector retrieval in RRF fusion?", "expected_complexity": "LOW"},
    {"query": "What does the get_optional_session function return?", "expected_complexity": "LOW"},
    {"query": "Where is the Firebase ID token verified?", "expected_complexity": "LOW"},
    {"query": "What port does the Streamlit frontend run on?", "expected_complexity": "LOW"},
    {"query": "Is Redis caching required or optional in this project?", "expected_complexity": "LOW"},

    # ---- MEDIUM: multi-function or single-subsystem reasoning ----
    {"query": "How does session creation and validation work in the FastAPI backend?", "expected_complexity": "MEDIUM"},
    {"query": "How does require_admin enforce role-based access control?", "expected_complexity": "MEDIUM"},
    {"query": "How does the hybrid search combine BM25 and vector retrieval results?", "expected_complexity": "MEDIUM"},
    {"query": "How does the reranker score and filter candidate chunks?", "expected_complexity": "MEDIUM"},
    {"query": "What happens when a user calls the /auth/logout endpoint?", "expected_complexity": "MEDIUM"},
    {"query": "How does api_call handle authenticated requests from the frontend?", "expected_complexity": "MEDIUM"},
    {"query": "How are BM25 and ChromaDB/AstraDB indices loaded on startup?", "expected_complexity": "MEDIUM"},
    {"query": "What's the difference between the public and admin views in Streamlit?", "expected_complexity": "MEDIUM"},

    # ---- HIGH: cross-file, cross-subsystem, architectural tracing ----
    {"query": "How does authentication and session management work across the frontend, API middleware, and database layers?", "expected_complexity": "HIGH"},
    {"query": "Trace a query from the Streamlit UI through the API to the final answer generation.", "expected_complexity": "HIGH"},
    {"query": "How do the BM25 retriever, vector retriever, RRF fusion, and reranker interact end to end?", "expected_complexity": "HIGH"},
    {"query": "How does the system handle a request from an unauthenticated user versus an admin, across the frontend and backend?", "expected_complexity": "HIGH"},
    {"query": "What would break if Redis caching were removed, and how does it interact with the rest of the architecture?", "expected_complexity": "HIGH"},
    {"query": "Compare how session state is managed on the frontend versus the backend, and how they stay in sync.", "expected_complexity": "HIGH"},
    {"query": "How does the project's deployment setup (Docker/EC2) relate to the FastAPI and Streamlit services running together?", "expected_complexity": "HIGH"},
]

UNANSWERABLE_SET = [
    {"query": "What database engine is used to store user passwords?", "expected_complexity": "LOW", "unanswerable": True},
    {"query": "How does the project handle payment processing?", "expected_complexity": "LOW", "unanswerable": True},
    {"query": "What rate limiting strategy is used on the public API endpoints?", "expected_complexity": "MEDIUM", "unanswerable": True},
    {"query": "How does the system support multi-tenant repository isolation?", "expected_complexity": "MEDIUM", "unanswerable": True},
    {"query": "What testing framework and CI/CD pipeline does this project use?", "expected_complexity": "MEDIUM", "unanswerable": True},
]

EVAL_SET = EVAL_SET + UNANSWERABLE_SET

ABSTENTION_PHRASES = [
    "cannot be determined", "not defined in", "does not contain",
    "no information", "not covered", "cannot find", "not present in the context",
    "elsewhere in the codebase", "not included in the provided context",
]

def is_abstention(answer: str) -> bool:
    a = answer.lower()
    return any(phrase in a for phrase in ABSTENTION_PHRASES)

def judge_answer(query: str, answer: str, context: str, llm) -> float:
    prompt = f"""Rate how well the ANSWER addresses the QUESTION using only the CONTEXT provided.
Score from 0.0 (wrong/irrelevant) to 1.0 (fully correct and complete).
Return ONLY a number.

QUESTION: {query}
CONTEXT: {context[:2000]}
ANSWER: {answer}

Score:"""
    try:
        response = llm.invoke([prompt])
        text = response.content or response.additional_kwargs.get("reasoning_content", "")
        match = re.search(r"[\d.]+", text)
        if not match:
            print(f"[judge] could not parse score from: {text!r}")
            return 0.0
        return max(0.0, min(1.0, float(match.group())))
    except Exception as e:
        print(f"[judge] scoring failed: {e}")
        return 0.0


def run_eval(eval_set, hybrid_search, reranker, llm, judge_llm=None, sleep_between=1.5):
    judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
    rows = []

    for item in eval_set:
        query = item["query"]
        is_unanswerable = item.get("unanswerable", False)

        # Baseline: single ragPipeline call, fixed top_k/top_n, no adaptive cutoff/trim
        t0 = time.monotonic()
        baseline_out = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker,
                                    llm=llm, top_k=20, top_n=5, use_adaptive=False, return_context=True)
        baseline_latency = time.monotonic() - t0
        baseline_tokens = count_tokens(baseline_out.get("context", ""))

        time.sleep(sleep_between)  # give AstraDB/Groq room to breathe between calls

        # Adaptive: full pipeline as built
        t0 = time.monotonic()
        adaptive_out = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker,
                                    llm=llm, return_context=True)
        adaptive_latency = time.monotonic() - t0
        adaptive_tokens = count_tokens(adaptive_out.get("context", ""))

        if is_unanswerable:
            baseline_score = score_unanswerable(baseline_out["answer"])
            adaptive_score = score_unanswerable(adaptive_out["answer"])
        else:
            baseline_score = judge_answer(query, baseline_out["answer"], baseline_out.get("context", ""), judge_llm)
            adaptive_score = judge_answer(query, adaptive_out["answer"], adaptive_out.get("context", ""), judge_llm)

        rows.append({
            "query": query,
            "unanswerable": is_unanswerable,
            "expected_complexity": item.get("expected_complexity"),
            "predicted_complexity": adaptive_out["complexity"],
            "baseline_tokens": baseline_tokens,
            "adaptive_tokens": adaptive_tokens,
            "token_reduction_pct": round(100 * (1 - adaptive_tokens / max(baseline_tokens, 1)), 1),
            "baseline_score": baseline_score,
            "adaptive_score": adaptive_score,
            "accuracy_retained_pct": round(100 * adaptive_score / max(baseline_score, 1e-6), 1),
            "baseline_latency_s": round(baseline_latency, 2),
            "adaptive_latency_s": round(adaptive_latency, 2),
        })
        print(f"✓ {query[:60]}... | tokens {baseline_tokens}->{adaptive_tokens} | score {baseline_score:.2f}->{adaptive_score:.2f}")

        time.sleep(sleep_between)  # pace between full queries too

    # ... rest (CSV write + summary) unchanged

    if rows:
        with open("eval_results.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

    answerable_rows = [r for r in rows if not r["unanswerable"]]
    unanswerable_rows = [r for r in rows if r["unanswerable"]]

    avg_token_reduction = (sum(r["token_reduction_pct"] for r in answerable_rows) / len(answerable_rows)) if answerable_rows else 0.0
    avg_accuracy_retained = (sum(r["accuracy_retained_pct"] for r in answerable_rows) / len(answerable_rows)) if answerable_rows else 0.0
    abstention_rate = (sum(r["adaptive_score"] for r in unanswerable_rows) / len(unanswerable_rows)) if unanswerable_rows else None

    print(f"\n===== SUMMARY =====")
    print(f"Avg token reduction (answerable queries): {avg_token_reduction:.1f}%")
    print(f"Avg accuracy retained (answerable queries): {avg_accuracy_retained:.1f}%")
    if abstention_rate is not None:
        print(f"Correct abstention rate (unanswerable queries): {abstention_rate*100:.1f}%")
    return rows


In [172]:
import time
test_query = "Where is the FastAPI entry point defined?"

baseline = ragPipeline(test_query, hybrid_search=hybrid_search, reranker=re_ranker,
                        llm=llm, top_k=20, top_n=5, use_adaptive=False, return_context=True)

time.sleep(1.5)  # Pace between LLM requests to respect Groq TPM rate limits

adaptive = ragPipeline(test_query, hybrid_search=hybrid_search, reranker=re_ranker,
                        llm=llm, return_context=True)

baseline_tokens = count_tokens(baseline.get("context", ""))
adaptive_tokens = count_tokens(adaptive.get("context", ""))

print("=== BASELINE (use_adaptive=False) ===")
print("Answer:", baseline["answer"])
print("Tokens:", baseline_tokens)
print("Chunks:", baseline["final_chunk_count"])

print("\n=== ADAPTIVE ===")
print("Answer:", adaptive["answer"])
print("Complexity:", adaptive["complexity"])
print("Tokens:", adaptive_tokens)
print("Chunks:", adaptive["final_chunk_count"])

print("\n=== DIFF CHECK ===")
print("Token reduction:", round(100 * (1 - adaptive_tokens / max(baseline_tokens, 1)), 1), "%")
print("Are baseline and adaptive identical?", baseline_tokens == adaptive_tokens and baseline["final_chunk_count"] == adaptive["final_chunk_count"])


Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
=== BASELINE (use_adaptive=False) ===
Answer: 
Tokens: 1595
Chunks: 5

=== ADAPTIVE ===
Answer: The FastAPI entry point is defined in the api_server.py file, where the app = FastAPI(...) instance is initialized and configured.
Complexity: MEDIUM
Tokens: 2590
Chunks: 7

=== DIFF CHECK ===
Token reduction: -62.4 %
Are baseline and adaptive identical? False


In [120]:
import requests, os

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
for m in resp.json()["data"]:
    if m["id"] == "allam-2-7b":
        print(m)

{'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096, 'hugging_face_id': 'ALLaM-AI/ALLaM-2.0-7B-Instruct', 'name': 'ALLaM-2-7b', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 4096, 'max_output_length': 4096, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['json_mode']}


In [110]:
import requests, os

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
for m in resp.json()["data"]:
    print(m["id"])

whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
groq/compound-mini
openai/gpt-oss-safeguard-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
groq/compound
canopylabs/orpheus-v1-english
qwen/qwen3.6-27b
allam-2-7b
canopylabs/orpheus-arabic-saudi


In [77]:
judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
test_score = judge_answer(
    "What language is this repository written in?",
    "Python.",
    "This is a Python codebase using FastAPI and Streamlit.",
    judge_llm
)
print(test_score)  # should print something like 1.0, not 0.0 with a [judge] error above it

1.0


In [ ]:
judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
results = run_eval(EVAL_SET, hybrid_search=hybrid_search, reranker=re_ranker, llm=llm, judge_llm=judge_llm)